In [0]:
# Conexão com ADLS Gen2
# Squad 1 - Dupla 4
# Tabelas: ecommerce_produtos e ecommerce_categorias 

In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow
dbutils.library.restartPython()

In [0]:
from dotenv import load_dotenv
import os

caminho_env = "/Workspace/Users/jeronimo.carlos104@gmail.com/estagio-empregadados-turma-2/.env"

load_dotenv(caminho_env, override=True)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

storage_account_name = "internshipdatalake"
container_name = "raw"

credential = ClientSecretCredential(tenant_id, client_id, client_secret)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential,
)

fs_client = service_client.get_file_system_client(container_name)

print("Conexão preparada com sucesso.")

In [0]:
referencia_path = "real-time-data"

arquivos_encontrados = []
for p in fs_client.get_paths(path=referencia_path):
    print(p.name)
    arquivos_encontrados.append(p.name)

print(f"\nTotal de itens encontrados: {len(arquivos_encontrados)}")

In [0]:
fs_client = service_client.get_file_system_client("raw")

print("Conteúdo do raw:")

for p in fs_client.get_paths():
    print(p.name, "| é diretório:", p.is_directory)

In [0]:
load_dotenv("/Workspace/Users/jeronimo.carlos104@gmail.com/estagio-empregadados-turma-2/.env", override=True)

In [0]:
print("Conteúdo da raiz do container 'raw':")
for p in fs_client.get_paths():
    print(p.name, "| é diretório:", p.is_directory)

In [0]:
arquivos = list(fs_client.get_paths(path="real-time-data"))
print(f"Itens em real-time-data: {len(arquivos)}")
for a in arquivos:
    print(a.name)

In [0]:
import pandas as pd
from io import BytesIO

# Encontra a pasta de timestamp mais recente dentro de real-time-data
def encontrar_pasta_mais_recente(fs_client, base_path="real-time-data"):
    pastas = set()
    for p in fs_client.get_paths(path=base_path, recursive=True):
        if not p.is_directory:
            pasta = "/".join(p.name.split("/")[:-1])
            pastas.add(pasta)
    return sorted(pastas)[-1]  # funciona porque o padrão YYYY/MM/DD/HHMMSS ordena certo como texto

pasta_mais_recente = encontrar_pasta_mais_recente(fs_client)
print("Lendo a partir de:", pasta_mais_recente)

def ler_parquet_do_adls(caminho_no_container):
    file_client = fs_client.get_file_client(caminho_no_container)
    download = file_client.download_file()
    conteudo = download.readall()
    return pd.read_parquet(BytesIO(conteudo))

df_pedidos_pd = ler_parquet_do_adls(f"{pasta_mais_recente}/ecommerce_pedidos.parquet")

df_pedidos = spark.createDataFrame(df_pedidos_pd)

print("Pedidos:", df_pedidos.count(), "linhas")
